In [44]:
from scraper import fetch_website_contents, fetch_website_links
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

In [3]:
load_dotenv()
key = os.getenv("OPENAI_API_KEY")

### Tight Prompt Vs Loose Prompts


In [6]:
# --------------Tight Prompt with One Shot Prompting----------------


from langchain_openai import ChatOpenAI
from IPython.display import Markdown, display
import time


link_system_prompt = '''
From list of links, select relevant https links to include in bochure.
Return ONLY VALID JSON.
{"links":[{"type":"about","url":"https://..."}]}
'''

# Defining Function for User Content 
def list_of_links(url):
    links = fetch_website_links(url)
    return "\n".join(links)

model = ChatOpenAI(
    model = "arcee-ai/trinity-mini:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=key,
    temperature=0
)

def select_relevant_link(url):

    messages = [
        {"role":"system","content":link_system_prompt},
        {"role":"user","content":list_of_links(url)}
    ]
    response = model.invoke(messages)
    return response.content

print(select_relevant_link("https://huggingface.co"))




{"links":[{"type":"endpoints","url":"https://endpoints.huggingface.co"},{"type":"discuss","url":"https://discuss.huggingface.co"},{"type":"status","url":"https://status.huggingface.co/"},{"type":"github","url":"https://github.com/huggingface"},{"type":"twitter","url":"https://twitter.com/huggingface"},{"type":"linkedin","url":"https://www.linkedin.com/company/huggingface/"},{"type":"apply","url":"https://apply.workable.com/huggingface/"}]}


In [32]:
#--------------Loose Prompt With Multi Shot Prompting------------------


link_system_prompt = '''
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links":[
        {"type":"about page", "url":"https://full.url/goes/here/about"},
        {"type":"careers page", "url":"https://another.full.url/careers"}
    ]
}
'''

def user_prompt_with_links(url):

    user_prompt = '''
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):
    '''
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


model = ChatOpenAI(
    model = "arcee-ai/trinity-mini:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=key,
    temperature=0
)

def select_relevant_link(url):

    messages = [
        {"role":"system","content":link_system_prompt},
        {"role":"user","content":user_prompt_with_links(url)}
    ]
    response = model.invoke(messages)
    return response.content

print(select_relevant_link("https://huggingface.co"))




{
  "links": [
    {"type": "about page", "url": "https://huggingface.co/brand"},
    {"type": "careers page", "url": "https://huggingface.co/join"},
    {"type": "pricing", "url": "https://huggingface.co/pricing"},
    {"type": "docs", "url": "https://huggingface.co/docs"},
    {"type": "blog", "url": "https://huggingface.co/blog"},
    {"type": "company", "url": "https://huggingface.co/enterprise"},
    {"type": "homepage", "url": "https://huggingface.co"}
  ]
}


### The Output Format

* Json Parser vs Pydantic's BaseModel

In [ ]:

import json

def fetch_content_from_relevant_links(url):
    content = fetch_website_contents(url)

    links_json = select_relevant_link(url)
    links_dict = json.loads(links_json)   # 🔥 FIX

    landing_page = f"### Landing Page {content}\n ##Relevant Links:\n"

    for link in links_dict["links"]:   # 🔥 Access correctly
        landing_page += f"\n\n ####Link: {link['type']}\n"
        landing_page += fetch_website_contents(link['url'])

    return landing_page

In [36]:
#--------------JSON PARSER---------------------

from langchain_core.output_parsers import JsonOutputParser


def select_relevant_links(url):
    messages = [
        {"role":"system","content":link_system_prompt},
        {"role":"user","content":user_prompt_with_links(url)[:5]}
    ]

    model = ChatOpenAI(
        model = "arcee-ai/trinity-mini:free",
        base_url = "https://openrouter.ai/api/v1",
        api_key = key
    )
    parser  =JsonOutputParser()
    chain = model | parser
    return chain.invoke(messages)
    

select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://example.com/about'},
  {'type': 'careers page', 'url': 'https://example.com/careers'}]}

In [38]:
type(select_relevant_links("https://huggingface.co"))

dict

In [57]:
#--------------------Pydantic----------------------


from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

class Link(BaseModel):
    type: str = Field(description="about, careers etc")
    url: str = Field(description="full https URL")

class LinkSchema(BaseModel):
    links: List[Link]

link_system_prompt = '''
Select 5-6 relevant links, exclude Terms of Service, Privacy, email links.
{"links":[{"type":"about","url":"https://..."}]}
'''

# Defining Function for User Content 
def list_of_links(url):
    links = fetch_website_links(url)[:10]
    return "\n".join(links)

model = ChatOpenAI(
    model = "z-ai/glm-4.5-air:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=key,
    temperature=0
)

local_model = ChatOllama(
    model= "phi3:mini",
    model_provider = "ollama",
    temperature=0,
    num_ctx = 2048
)

def select_relevant_link(url):

    messages = [
        {"role":"system","content":link_system_prompt},
        {"role":"user","content":list_of_links(url)}
    ]
    structured_model = local_model.with_structured_output(LinkSchema)
    response = structured_model.invoke(messages)
    return response


select_relevant_link("https://www.ferventlearning.com/category/finance/investment-analysis")

LinkSchema(links=[Link(type='about', url='https://www.ferventlearning.com'), Link(type='courses', url='https://www.ferventlearning.com/courses/'), Link(type='resource-hub', url='https://www.ferventlearning.com/resource-hub/'), Link(type='articles', url='https://www.ferventlearning.com/articles/'), Link(type='all-access-pass', url='https://www.ferventlearning.com/all-access-pass/')])

In [58]:
def fetch_content_from_relevant_links(url):
    content = fetch_website_contents(url)
    data = select_relevant_link(url)

    landing_page = f"### Landing Page {content}\n ##Relevant Links:\n"

    for link in data.links:   # ✅ No JSON parsing needed
        landing_page += f"\n\n ####Link: {link.type}\n"
        landing_page += fetch_website_contents(link.url)

    return landing_page

fetch_content_from_relevant_links("https://www.ferventlearning.com/category/finance/investment-analysis")

'### Landing Page Investment Analysis Archives - Fervent | Finance Courses, Investing Courses\n\nSkip to main content\nSkip to footer\nFervent | Finance Courses, Investing Courses\nRigorous Courses, Backed by Research, Taught with Simplicity.\nHome\nCourses\nResource Hub\nArticles\nAll Access Pass\nInvestment Analysis\nExplore articles in Investment Analysis\n12 Practical, Applied Finance Data Science Projects – Beginner, Intermediate, and Advanced Levels\nAugust 17, 2022\nBy\nVash\nLeave a Comment\nDo you want to build your financial data science skills? Whether you\'re doing this to become a Financial Data Scientist, or to leverage the power of Financial Data Science for your own investments - …\nContinue Reading\nabout 12 Practical, Applied Finance Data Science Projects – Beginner, Intermediate, and Advanced Levels\n→\nFiled Under:\nFinance\n,\nFinancial Data Science\n,\nInvestment Analysis\nHow to Calculate Portfolio Beta Manually & In Excel®\nJune 1, 2022\nBy\nVash\nLeave a Commen